In [1]:
%load_ext autoreload
%autoreload 2

# Finding the patient: understanding the scale of real-world problems

We have a real-world task for you all! 

You are assigned a rare disease and a cohort of patients, each described by a set of [Human Phenotype Ontology (HPO)](https://hpo.jax.org) terms. Exactly one patient has that particular rare disease. 
The task is to figure out which patient has that disease!


The central goal of this assignment is to understand **how literature research systems reach their conclusions, where they fail, and how to investigate those failures**. You should be able to explain how the system found its evidence, interpreted it, and used it to produce a ranking.

You will start with retrieval-augmented generation (RAG): retrieving relevant passages and asking a model to answer using those passages. 

Then you will then work through **GRILL** and inspect its intermediate outputs to understand what each contributes.

**GRILL** searches and reads the literature while maintaining a *belief ledger*: hypotheses, evidence for and against them, and unresolved questions. You will trace how it moves from an initial question to competing hypotheses and chooses what to investigate next.


To understand teh scale of the problem, we already run the analysis on the rag systems, and GRILL and find that RAG is only at 39% accuracy and GRILL at 53%. 

| System | top@1 | top@2 |
|--------|------:|------:|
| RAG    | 39%   | 53%   |
| GRILL  | 55%   | 70%   |

Note: we ran RAG with web-search results (providing top-6 results) and running GRILL for 2 rounds (<$2)

All the LLM runs, including Grill have been stored for you -- you don't have to use credits.

### How to work through this notebook

Cells marked **TODO** are yours. The remaining cells run as given and expose the stages you will inspect.

---
# Part 0 — Setup

Run the setup cells with the project's environment. Read GRILL.md and SLIDERS.md, and locate the relevant implementation. The notebook identifies vendor/grill/src/ledger.py

In [2]:
import os, sys, json, math, textwrap, warnings
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path.cwd()
if ROOT.name == "notebook":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import rdx
from rdx import extract as X, grill as G

print("python     ", sys.version.split()[0])
print("root       ", ROOT)
print("GRILL at   ", G.GRILL_ROOT)

python      3.12.3
root        /mnt/data/oval/assignment-cs224v
GRILL at    /mnt/data/oval/assignment-cs224v/vendor/grill


### The disease you were assigned

**TODO 0:** You are given a unique disease to investigate.

In [3]:
# ---------------------------------------------------------------------
# YOUR DISEASE. Change this line, and nothing else, to re-target the
# notebook. It must match the disease you ran setup_disease.py for.
# ---------------------------------------------------------------------
# TODO: change this to your disease ID, e.g. "OMIM:620851"
DISEASE = "OMIM:620851"
# ---------------------------------------------------------------------

from rdx import load_hpo
from rdx.phenopackets import PhenopacketStore
from rdx.disease import DiseaseBrief

hpo   = load_hpo("data")                              # ~4 s
store = PhenopacketStore.load(ontology=hpo.ontology)  # ~7 s
brief = DiseaseBrief.resolve(DISEASE, store, hpo)

TARGET = DISEASE            # the rest of the notebook uses this name

print(brief.summary())

ReNU syndrome  (OMIM:620851)
  gene                 RNU4-2  RNA, U4 small nuclear 2
                       snRNA, 12q24.23, NCBI Gene 26834
                       also called NEDHAFA, RENU, RNU4-1B, RNU4B1, RNU4C
  curated HPO terms    198
  case reports         3 PMIDs
  curated from         3 references
  literature query     "RNU4-2" OR "ReNU syndrome"


---

# 1 Using RAG

Retrieval Augmented Generation (RAG) is an artificial intelligence technique that improves LLMs responses by fetching relevant data from an external knowledge base before generating an answer.

We will start with obtaining search query results. Then we will feed the LLM with the fetched result and ask it to describe the disease. Finally, we will ask it to rank patients.

## 1.1 Searching with API

The initial search returns titles, URLs, and short snippets. It does not mean the model has read those pages or their full papers. A search snippet can omit qualifications or combine surrounding material in misleading ways.

In [4]:
import urllib.request

## Step 1: First build your query

query = f"{brief.gene} gene {brief.label}"

## Step 2: Make an API call to the web search API
# We have already pre-fetched the results for your disease.
sim = json.loads((ROOT / "assignments" / DISEASE.replace(":", "_") / "rag_simulation.json").read_text())
hits = sim["search"]["hits"]

## Step 3: Print the search query results
print(f"query sent to the search engine: {sim['search']['query']!r}")
print(f"generated {sim['generated']}\n")
print(f"the {len(hits)} results, exactly as returned:\n")
for i, h in enumerate(hits, 1):
    print(f"[{i}] {h['title']}")
    print(f"    {h['link']}")
    print(textwrap.fill(h.get('snippet', ''), 90, initial_indent="    ", subsequent_indent="    "))
    print()

query sent to the search engine: 'RNU4-2 gene ReNU syndrome'
generated 2026-09-28 20:48

the 6 results, exactly as returned:

[1] RNU4-2–Related Autosomal Dominant Neurodevelopmental ...
    https://www.ncbi.nlm.nih.gov/books/NBK622730/
    RNU4-2–related autosomal dominant neurodevelopmental disorder (ReNU syndrome) is
    characterized by moderate-to-severe developmental delay (DD) ...

[2] De novo variants in the RNU4-2 snRNA cause a frequent ...
    https://www.nature.com/articles/s41586-024-07773-7
    by Y Chen · 2024 · Cited by 179 — Biallelic variants in the noncoding RNA gene RNU4-2
    cause a recessive neurodevelopmental syndrome with distinct white matter changes.
    Article ...

[3] RNU4-2 / ReNU Syndrome | Join Research Today for Hope
    https://www.renusyndrome.org/
    ReNU syndrome is a spliceosomal disorder with a defined mutational hotspot and growing
    global dataset, creating a uniquely actionable opportunity for ...

[4] RNU4-2 | UDN
    https://undiagnosed.hm

**TODO 1.1**: Write the about what kinds of urls do you see? Are they databases, published articles, blogpost, news reporting etc.

## 1.2 What a context to the LLM looks like

In the cell above you see six titles, six links, six snippets of a sentence or two. That is all the
retrieval step returns; nobody has read a paper yet. 

The next cell turns it into the **context** for the model

A **system** message gives the LLM high-priority instructions about how it should behave. A **user** message contains the user’s request or input that the LLM should respond to.

This is the task we will give the LLM:

```
Task: In about 150 words, describe the gene RNU4-2 and its relation to ReNU syndrome:
what the gene product is, what it does in the cell, and how variants in it cause disease.
Every sentence needs a citation like [2].
```

In [5]:
"""A LLM call asking it to generate describe the disease based on 
retrieved information. (Beware, these might contain hallucinations!)
"""

# Build the context
context = sim["context"]

# A LLM call usually consists of System Message, User Message and the Assistant's reply.
# System message provides the high-level prompt to the llm for a conversation.

messages = [
    {"role": "system",
     "content": "Use only the numbered sources given. Cite each claim as [n]. "
                "If the sources do not support a statement, say it is not covered."},
    {"role": "user",
     "content": f"Sources:\n\n{context}\n\n"
                f"Task: In about 150 words, describe the gene {brief.gene} and its "
                f"relation to {brief.label}: what the gene product is, what it does "
                f"in the cell, and how variants in it cause disease. Every sentence "
                f"needs a citation like [2]."},
]
for m in messages:
    print(f"--- {m['role']} ({len(m['content'])} chars) ---")
    print(textwrap.indent(m["content"], "  "))
    print()

--- system (126 chars) ---
  Use only the numbered sources given. Cite each claim as [n]. If the sources do not support a statement, say it is not covered.

--- user (1815 chars) ---
  Sources:

  [1] RNU4-2–Related Autosomal Dominant Neurodevelopmental ...
  https://www.ncbi.nlm.nih.gov/books/NBK622730/
  RNU4-2–related autosomal dominant neurodevelopmental disorder (ReNU syndrome) is characterized by moderate-to-severe developmental delay (DD) ...

  [2] De novo variants in the RNU4-2 snRNA cause a frequent ...
  https://www.nature.com/articles/s41586-024-07773-7
  by Y Chen · 2024 · Cited by 179 — Biallelic variants in the noncoding RNA gene RNU4-2 cause a recessive neurodevelopmental syndrome with distinct white matter changes. Article ...

  [3] RNU4-2 / ReNU Syndrome | Join Research Today for Hope
  https://www.renusyndrome.org/
  ReNU syndrome is a spliceosomal disorder with a defined mutational hotspot and growing global dataset, creating a uniquely actionable opportunity for .

## 1.3 Now lets ask an LLM about the disease by providing it information from search results

This is the answer generated through RAG.

In [6]:
answer = sim["gene_rag"]["answer"]
print(f"tokens in: {sim['gene_rag']['tokens_in']}   out: {sim['gene_rag']['tokens_out']}\n")
print(textwrap.fill(answer, 90))

tokens in: 588   out: 1584

*RNU4-2* is a noncoding RNA gene that encodes a small nuclear RNA (snRNA) product [2].
While its complete mechanism of action in the cell is not explicitly detailed in the
provided sources, the condition it causes is identified as a spliceosomal disorder,
implicating it in spliceosome function [3].   ReNU syndrome is caused by pathogenic
variants, or specific changes in the DNA sequence, of the *RNU4-2* gene [6]. Disease-
causing alterations frequently occur as *de novo* mutations situated at a defined
mutational hotspot [3, 4]. Primarily recognized as an autosomal dominant
neurodevelopmental disorder, ReNU syndrome presents with moderate-to-severe developmental
delay and is also known as neurodevelopmental disorder with hypotonia, brain anomalies,
distinctive facies, and absent language (NEDHAFA) [1, 5]. Additionally, biallelic variants
in *RNU4-2* have been reported to cause a recessive neurodevelopmental syndrome marked by
distinct white matter changes [2

## 1.4 What happens when the don't provide this context?

Now we ask LLM the same question, but ask it to generate answerr through its memory.

In [7]:
print("the question, with no sources:\n")
print(textwrap.indent(sim["gene_closed"]["messages"][0]["content"], "  "))
print("\nthe answer:\n")
print(textwrap.fill(sim["gene_closed"]["answer"], 90))

the question, with no sources:

  In about 150 words, describe the gene RNU4-2 and its relation to ReNU syndrome: what the gene product is, what it does, how variants cause disease. Give the year the disease was first described and cite the paper.

the answer:

The ***RNU4-2*** gene encodes U4 small nuclear RNA (snRNA), an essential non-coding RNA
component of the major spliceosome. In normal cellular function, U4 snRNA base-pairs with
U6 snRNA to form the U4/U6 di-snRNP complex. This complex acts as a regulatory chaperone,
maintaining U6 in an inactive conformation until the spliceosome is properly assembled on
pre-mRNA introns for splicing.  ReNU syndrome is a neurodevelopmental disorder
characterized by severe intellectual disability, microcephaly, speech loss, and motor
impairment. It is caused by recurrent, *de novo* mutations clustering within a highly
conserved 18-base-pair region (stem II) of *RNU4-2* (most commonly single-base insertions
like n.64_65insT). These variants disru

**TODO 1.2:** After reading the two response in 1.3 and 1.4, What is the stylistic difference betweent the two responses? What is the content difference in the two responses? Which response do you prefer? 

# 2 The cohort: real published patients

Now that we have some information about the disease, lets look into the patients data.

We use [GA4GH Phenopacket
Store](https://github.com/monarch-initiative/phenopacket-store), a database with 10,377
individuals across 780 Mendelian diseases, each curated from a published case
report, each carrying HPO terms, an OMIM diagnosis, the causative variant and
the PMID it came from.

A **disease phenotype** is the set of observable signs, symptoms, physical traits, or lab results caused by a medical condition.a

**Explicit negatives.** 87% of these patients have at least one *excluded*
finding. For example: A clinician looked for phenotype: nystagmus and recorded its absence.

## 2.1 Let's look at the patient cohort

In the output below you will see fields `obs` and `excl`.

`obs` means observered phenotypes
`excl` means explcuded phenotypes (explicit negatives)

In [8]:
from rdx import render_patient, cohort_overlap
from rdx.assignments import load_assignment

disease, cohort = load_assignment(DISEASE)
fields = sorted(json.loads((ROOT / "assignments" / DISEASE.replace(":", "_") / "cohort.json").read_text())[0])
print(f"{len(cohort)} patients, fields on disk: {fields}\n")
print(f"{'id':<5} {'obs':>4} {'excl':>5} {'age':>6} {'sex':<7}")
for p in cohort.patients:
    print(f"{p.patient_id:<5} {len(p.terms):4d} {len(p.excluded):5d} "
          f"{str(p.age_years):>6} {str(p.sex):<7}")

10 patients, fields on disk: ['age_years', 'excluded', 'patient_id', 'sex', 'terms']

id     obs  excl    age sex    
P01      9    37    3.4 female 
P02     10    90    1.6 female 
P03     24     3    2.3 male   
P04      8     0   11.0 female 
P05     16     5   11.0 male   
P06     31    40    1.7 male   
P07      9     1    6.0 male   
P08     13    23    8.0 female 
P09      8     7    0.7 female 
P10     24   130    5.0 male   


## 2.2 What one patient actually looks like to you.

Now lets take a closer look at one of the patients information. Remember all these phenotypes are already canonicalized. 

In NLP, **canonicalization** means converting different forms of the same underlying text into one standard, consistent representation. For example, `"USA"`, `"U.S.A."`, and `"United States"` might all be mapped to a single canonical form like `"United States"`. Or, `“Homo sapiens”` and `“human”` look like different labels, but they refer to the same species.

In [9]:
# feel free to change the index and check other patients
p = cohort.patients[2]
print(render_patient(p, hpo.ontology))
print("\n  recorded ABSENT:")
for t in p.excluded[:8]:
    print(f"    - {t}  {hpo.label(t)}")
if len(p.excluded) > 8:
    print(f"    ... and {len(p.excluded) - 8} more")

### Patient P03
- age: 2.3 years
- sex: male
- phenotypes:
    - HP:0010804  Tented upper lip vermilion
    - HP:0000179  Thick lower lip vermilion
    - HP:0000341  Narrow forehead
    - HP:0000294  Low anterior hairline
    - HP:0000293  Full cheeks
    - HP:0000286  Epicanthus
    - HP:0000582  Upslanted palpebral fissure
    - HP:0000527  Long eyelashes
    - HP:0000664  Synophrys
    - HP:0003196  Short nose
    - HP:0011451  Primary microcephaly
    - HP:0001276  Hypertonia
    - HP:0000316  Hypertelorism
    - HP:0000639  Nystagmus
    - HP:0000365  Hearing impairment
    - HP:0001263  Global developmental delay
    - HP:0001249  Intellectual disability
    - HP:0002353  EEG abnormality
    - HP:0032792  Tonic seizure
    - HP:0001272  Cerebellar atrophy
    - HP:0033725  Thin corpus callosum
    - HP:0033454  Tube feeding
    - HP:0001903  Anemia
    - HP:0001873  Thrombocytopenia

  recorded ABSENT:
    - HP:0002307  Drooling
    - HP:0001252  Hypotonia
    - HP:0002317  Unste

# 3 Patient matching with RAG

We now provide the patient data to the LLM and try to see what response do we get. But first we build the context.

In [11]:
rank_run = json.loads((ROOT / "assignments" / DISEASE.replace(":", "_") / "rag_match.json").read_text())
rank_messages = rank_run["messages"]

for m in rank_messages:
    print(f"--- {m['role']} ({len(m['content']):,} chars) ---")
    body = m["content"]
    print(textwrap.indent(body if m["role"] == "system" else body[:1500] + "\n  [... snippets and the remaining patients ...]", "  "))
    print()

This time the model is asked for a RANKED top five, in a fixed JSON shape, so the
answer can be scored. Same six snippets; the patients now include what was looked
for and recorded ABSENT, which the report-reading step later will also see.

--- system (330 chars) ---
  You are a clinical geneticist. You are given evidence about one rare disease and a cohort of patients described by HPO terms. Exactly one patient has the disease. Use only that evidence as your knowledge of the disease. Be explicit about which findings in the report drove each placement, and about findings that argue against it.

--- user (8,392 chars) ---
  WEB SEARCH RESULTS FOR ReNU syndrome (RNU4-2, OMIM:620851):

  [1] RNU4-2–Related Autosomal Dominant Neurodevelopmental ...
  https://www.ncbi.nlm.nih.gov/books/NBK622730/
  RNU4-2–related autosomal dominant neurodevelopmental disorder (ReNU syndrome) is characterized by moderate-to-severe developmental delay (DD) ...

  [2] De novo variants in the RNU4-2 snRNA cause

## 3.1 Running LLMs.

You will observe that in some cases, running the LLM multple times over the same context gives different results! Here we run the LLM three times on the same patient cohort.

You will observe that for each patient, the model has a for and against, and an assigned probability.

In [16]:
import pandas as pd
rows = []
for r in rank_run["rounds"]:
    for pos, c in enumerate(r["ranking"][:5], 1):
        rows.append({"round": r["round"], "position": pos, "patient": c["patient_id"],
                     "probability": c["probability"], "for": c["for"], "against": c["against"]})
ranking = pd.DataFrame(rows).set_index(["round", "position"])

display(ranking.style
          .set_properties(subset=["for", "against"],
                          **{"white-space": "normal", "text-align": "left", "max-width": "420px"})
          .format({"probability": "{:.2f}"})
          .set_caption(f"three rounds, same prompt: the model's top five for {brief.label}"))

**TODO 3.1:** Did repeated runs agree? They if do not disagree, do their reasoning match, please describe. If they do not match, how does their reasoning differ?

**TODO 3.2:** What is the differentiating factor in each round between the first and the second ranks?

# 4 GRILL: Using Hypotheses to Guide Search

A basic literature-search system can find papers and return a list of links. GRILL goes a step further: it uses what it finds to decide **what to search for next**.

Instead of treating search as a one-time query, GRILL treats it as an iterative process. It starts with questions, searches for relevant information, forms possible explanations or claims, and then looks for evidence that supports or contradicts those claims. New evidence can lead to new hypotheses and new questions. This process continues until GRILL either reaches a sufficiently clear answer or runs out of its search budget.

You can think of GRILL as maintaining a structured research notebook while it searches.

GRILL keeps track of three main types of records:

| Record type | What it means |
|---|---|
| **Direction** | An open question or area that still needs investigation. Directions form the search **frontier**: the set of things GRILL could explore next. The `EXPLORE` step spends search budget on these directions. |
| **Hypothesis** | A possible claim or explanation that GRILL wants to test. A hypothesis moves through several stages: `proposed → screened → pooled` or `binned`, depending on what GRILL learns about it. |
| **Evidence** | A specific observation from a source that is connected to a hypothesis. Evidence is labeled as either **supporting** or **contradicting** the hypothesis. |

For example, suppose GRILL is investigating the question:

> Why do some large language models perform better when given examples in the prompt?

A **direction** might be: “Does the number of examples matter?”

From its search, GRILL might create the **hypothesis**:

> Providing more examples improves performance up to a certain point.

It could then collect **supporting evidence** from one study showing that performance improves with additional examples, and **contradicting evidence** from another study showing that too many examples can reduce performance.

Based on this evidence, GRILL may refine the hypothesis or create a new direction, such as:

> Does the usefulness of additional examples depend on the model size?

This is the main idea behind GRILL: **search produces hypotheses, hypotheses guide further search, and evidence is used to evaluate those hypotheses.**

### What to examine in this part

There are five aspects of GRILL to investigate. Each one corresponds to a TODO in the exercise.

1. **Where do directions come from?**  
   Understand how GRILL decides which open questions should be added to the search frontier.

2. **What does a hypothesis actually contain?**  
   Look at how hypotheses are represented in the system and what information GRILL stores about each one.

3. **What happens if the verifier only searches for supporting evidence?**  
   Normally, a good research process should look for both evidence that supports a hypothesis and evidence that challenges it. Investigate how GRILL's behavior changes if it only tries to confirm its hypotheses.

4. **What happens if deep hypothesis testing is skipped?**  
   Examine what GRILL loses if hypotheses are accepted or rejected without a more thorough search for evidence.

5. **How is the search budget allocated?**  
   GRILL cannot search indefinitely. Investigate how it decides which directions or hypotheses deserve more search effort and when to stop exploring them.

### Files to use

`GRILL.md` explains the intended design and behavior of the system.

`vendor/grill/src/ledger.py` contains the implementation of the ledger that keeps track of directions, hypotheses, evidence, and their state.

Use `GRILL.md` to understand **what GRILL is supposed to do**, and use `ledger.py` to understand **how those ideas are represented in code**.

## 4.1 Summary of the GRILL run

In the output you will see how many directions and hypothesis did grill come up with. How many evidences it found and how much did it cost us.

In [18]:
G.add_grill_to_path()
import shutil
from src.ledger import Ledger, SourceRef, POOLED, BINNED, SCREENED, PROPOSED
from src.orchestrator import Orchestrator, Config
from src import metrics

GRILL_MODE = "read"          # "read" | "stub" | "live"

if GRILL_MODE == "read":
    # A finished run, shipped with the assignment. Nothing executes in it, so
    # it is safe inside the repository -- unlike a live run, see 2.6.
    RUN_DIR = ROOT / "assignments" / DISEASE.replace(":", "_") / "grill_run"
else:
    RUN_DIR = G.run_dir_for(f"renu_{GRILL_MODE}")

run = G.GrillRun(RUN_DIR)
print(run)

<GrillRun grill_run: 58 directions, 18 hypotheses (7 pooled), 108 evidence, $0.57>


### 4.2 Where Directions Come From

In GRILL, a **direction** is an open question that the system has decided is worth spending part of its search budget on.

Directions can come from four different sources. Looking at where they came from helps you understand how the search process developed.

| Origin | Where it comes from |
|---|---|
| **Seed** | Created during `INIT`, before any literature search happens. These come from the PRIOR executor's `open_questions` and serve as the starting questions for the search. |
| **Agent** | Created during `EXPLORE`. After investigating one funded direction, GRILL may discover new questions that are worth following up on. |
| **Corrective** | Created when a hypothesis is found to be conflicted or refuted. Instead of simply discarding the failed hypothesis, GRILL turns the problem into a new question to investigate. These directions have `tests_hypothesis_id` set, which links them back to the hypothesis they are trying to resolve. |
| **Human** | Added directly by a person through GRILL's steering mechanism. These directions have `origin == "human"`. |

The **corrective directions** are especially important because they show how GRILL responds to failure.

Suppose GRILL proposes a hypothesis such as:

> “Treatment X improves symptom Y.”

During deeper testing, it might find strong evidence that contradicts this claim. A simpler search system might just reject the hypothesis and move on.

GRILL does something different. It creates a new direction such as:

> “Why is the evidence about Treatment X and symptom Y conflicting?”

In other words, **a failed hypothesis does not disappear from the research process. It becomes a new question that GRILL can investigate.**

This is an important part of GRILL's design: negative or conflicting evidence can change the direction of the search rather than simply ending it.

## 4.3 Now we will look at the initial questions generated by GRILL
`promise`: This is the confidence that the LLM has on this direction. Note LLMs are known to not be well calibrated. 

`funded`: This means how many directions is the agent pursuing.

[1] Xiong, Miao, et al. "Can llms express their uncertainty? an empirical evaluation of confidence elicitation in llms." International Conference on Learning Representations. Vol. 2024. 2024.

In [19]:
import pandas as pd
from collections import Counter

D = run.directions_table()
ledger_raw = run.ledger_json

def classify(d):
    if d["tests"]:
        return "corrective"
    if d["origin"] == "human":
        return "human"
    if d["parent"] is None and "seed" in (ledger_raw["directions"][d["id"]].get("rationale") or ""):
        return "seed"
    return "agent"

for d in D:
    d["kind"] = classify(d)
print(f"{len(D)} directions:", dict(Counter(d["kind"] for d in D)))

# One ranking, not three samples. Promise is the number the allocator sorts
# by, so this is the order in which directions were offered money.
directions = (pd.DataFrame(D)
              .assign(funded=lambda t: t.spent > 0)
              .sort_values("promise", ascending=False)
              .reset_index(drop=True))
directions.index = directions.index + 1
directions.index.name = "rank"
ranked = directions.to_dict("records")          # later cells iterate this

cols = ["id", "kind", "promise", "spent", "n_hypotheses", "funded", "question"]
print(f"{int(directions.funded.sum())} of {len(directions)} directions were ever funded. "
      f"Promise decides which; the rest are the frontier that never got a turn.\n")

def show_directions(frame):
    """Full question text, wrapped; funded rows highlighted."""
    return (frame[cols].style
            .set_properties(subset=["question"],
                            **{"white-space": "normal", "text-align": "left", "max-width": "640px"})
            .format({"promise": "{:.2f}", "spent": "${:.2f}"})
            .apply(lambda r: ["background-color: #fff3cd" if r.funded else "" for _ in r], axis=1))

show_directions(directions)

58 directions: {'seed': 4, 'agent': 43, 'corrective': 11}
4 of 58 directions were ever funded. Promise decides which; the rest are the frontier that never got a turn.



,id,kind,promise,spent,n_hypotheses,funded,question
rank,,,,,,,
1,d1,seed,0.95,$0.02,3,True,"What are the exact numerator/denominator frequencies for specific HPO terms across the first published cohorts (Greene et al. Nature 2024, Chen et al. Nature 2024)?"
2,d19,agent,0.95,$0.02,3,True,"What are the exact numerator and denominator frequencies for specific brain MRI malformations (e.g., corpus callosum hypoplasia, delayed myelination, simplified gyral pattern) across Greene et al., Chen et al., and Barbour et al. cohorts?"
3,d2,seed,0.92,$0.01,3,True,What specific dysmorphic features or clinical signs distinguish ReNU syndrome from Rett syndrome (MECP2) and Angelman syndrome (UBE3A/15q11-q13)?
4,d3,seed,0.90,$0.01,4,True,Are there clinical features that have been consistently evaluated and confirmed absent across cohorts (pertinent negatives)?
5,d55,agent,0.88,$0.00,0,False,What is the exact prevalence and sensitivity of congenital talipes equinovarus versus postnatal hip dysplasia across systematically phenotyped RNU4-2 multi-center cohorts?
6,d53,agent,0.88,$0.00,0,False,"How accurately can automated computer-vision facial phenotyping tools (e.g., Face2Gene/GestaltMatcher) discriminate ReNU syndrome from Rett syndrome (MECP2) and Angelman syndrome (UBE3A)?"
7,d50,agent,0.85,$0.00,0,False,"What proportion of RNU4-2 patients present with normal brain MRI, and does normal callosal anatomy correlate with specific non-hotspot genotypes (such as 5' stem-loop variants)?"
8,d41,agent,0.85,$0.00,0,False,What are the exact patient-level numerators and denominators for corpus callosum hypoplasia versus simplified gyral pattern specifically in the Greene et al. (2024) and Barbour et al. cohorts?
9,d46,agent,0.85,$0.00,0,False,"What is the true population-wide penetrance of neuroimaging abnormalities in ReNU syndrome when stratifying by variant position (e.g., T-loop n.64_65insT vs. stem III vs. maternal/milder variants)?"


## 4.4 Now let's take a closer look at all generated hypotheses

Each hyoptheses gets a verdict of:

**Supported**: If on investigation we find evidence that supports the hypothesis.

**Refuted**: If all the evidences found for the hyopthesis refute the hypothesis.

**Conflicted**: If there are multiple evidences that support and refute the hypothesis.

In [20]:
H = run.hypotheses_table()

rank_of = {r.id: int(i) for i, r in directions.iterrows()}
question_of = dict(zip(directions.id, directions.question))

hyps = (pd.DataFrame(H)
        .rename(columns={"direction": "direction_id",
                         "n_support": "evidence_for",
                         "n_against": "evidence_against",
                         "text": "hypothesis"})
        .assign(direction_rank=lambda t: t.direction_id.map(rank_of),
                direction=lambda t: t.direction_id.map(question_of))
        .sort_values(["direction_rank", "confidence"], ascending=[True, False])
        .set_index(["direction_rank", "direction_id", "id"]))

hcols = ["status", "verdict", "confidence", "evidence_for", "evidence_against",
         "attempts", "hypothesis", "direction"]

def show_hypotheses(frame):
    """Every hypothesis in full, grouped under the direction that proposed it.
    Green: pooled. Red: refuted. Unshaded: binned as conflicted."""
    return (frame[hcols].style
            .set_properties(subset=["hypothesis", "direction"],
                            **{"white-space": "normal", "text-align": "left", "max-width": "520px"})
            .format({"confidence": "{:.2f}"})
            .apply(lambda r: ["background-color: #e6f4ea" if r.status == "pooled"
                              else "background-color: #fdecea" if r.verdict == "refuted"
                              else "" for _ in r], axis=1))

print(f"{len(hyps)} hypotheses from {hyps.index.get_level_values('direction_id').nunique()} directions, "
      f"ordered by the rank of the direction that proposed them")
show_hypotheses(hyps)

18 hypotheses from 4 directions, ordered by the rank of the direction that proposed them


### 4.5 What to do when a hypothesis is wrong?

When a hypothesis is wrong, we convert them into new directions. Every `"binned"` claim becomes a new question.

In [17]:

hyp_of = {h["id"]: h for h in H}
rank_of = {r.id: int(i) for i, r in directions.iterrows()}

rows = []
for d in D:
    if not d["tests"]:
        continue
    h = hyp_of.get(d["tests"], {})
    rows.append({
        "failed_hypothesis": d["tests"],
        "its_verdict":       h.get("verdict"),
        "its_confidence":    h.get("confidence"),
        "evidence_for":      h.get("n_support"),
        "evidence_against":  h.get("n_against"),
        "the_claim":         h.get("text"),
        "why_binned":        h.get("bin_reason"),
        "new_direction":     d["id"],
        "rank":              rank_of.get(d["id"]),
        "promise":           d["promise"],
        "funded":            d["spent"] > 0,
        "the_question":      d["question"],
    })

if not rows:
    print("No corrective directions in this run: nothing was contested.")
else:
    corrective = (pd.DataFrame(rows)
                  .sort_values("rank")
                  .set_index(["failed_hypothesis", "new_direction"]))
    print(f"{len(corrective)} binned hypotheses became directions. "
          f"{int(corrective.funded.sum())} of them were funded. "
          f"The best-ranked sits at rank {int(corrective['rank'].min())} of {len(directions)}.")

ccols = ["its_verdict", "its_confidence", "evidence_for", "evidence_against",
         "rank", "promise", "funded", "the_claim", "why_binned", "the_question"]

def show_corrective(frame):
    return (frame[ccols].style
            .set_properties(subset=["the_claim", "why_binned", "the_question"],
                            **{"white-space": "normal", "text-align": "left", "max-width": "420px"})
            .format({"its_confidence": "{:.2f}", "promise": "{:.2f}"})
            .apply(lambda r: ["background-color: #fff3cd" if r.funded else "" for _ in r], axis=1))

show_corrective(corrective)

11 binned hypotheses became directions. 0 of them were funded. The best-ranked sits at rank 10 of 58.


,,its_verdict,its_confidence,evidence_for,evidence_against,rank,promise,funded,the_claim,why_binned,the_question
failed_hypothesis,new_direction,,,,,,,,,,
h14,d45,conflicted,0.72,5,2,10,0.85,False,"Overall neuroimaging abnormality penetrance in ReNU syndrome exceeds 90% in systematically imaged cohorts (specifically 41/45 in Chen et al.), but findings are predominantly composed of non-specific white matter reduction, delayed myelination (HP:0002188), and ventriculomegaly (HP:0002119) rather than major cortical migration defects.","contested (5 for / 2 against): Penetrance estimate discrepancy: Chen et al. (2024) and Orphanet cite 91% (41/45) and Okamoto et al. (2025) cite 100% (9/9), whereas GeneReviews (Barbosa et al., 2026) summarizes abnormal brain MRI penetrance across cohorts at 85%, falling below the asserted >90% threshold.; Presence of cortical migration and gyration defects: Valenzuela & Codina (2024) and Orphanet report frontal simplified gyral patterns and heterotopias, indicating that while white matter reduction and ventriculomegaly predominate, migration and sulcation abnormalities are not completely absent.","Resolve the conflict over: Overall neuroimaging abnormality penetrance in ReNU syndrome exceeds 90% in systematically imaged cohorts (specifically 41/45 in Chen et al.), but findings are predominantly composed of non-specific white matter reduction, delayed myelination (HP:0002188), and ventriculomegaly (HP:0002119) rather than major cortical migration defects."
h17,d56,conflicted,0.45,4,1,12,0.85,False,"The structural brain MRI triad of corpus callosum hypoplasia/thinning, ventriculomegaly, and reduced white matter volume is present in over 85% of ReNU syndrome patients past infancy, providing an objective neuroradiological separator from Rett syndrome and Angelman syndrome, which typically show normal neuroimaging or non-specific cerebral volume reduction.","contested (4 for / 1 against): GeneReviews sets the overall rate of *any* brain MRI abnormality at 85%, which contradicts the hypothesis that the full triad specifically occurs in >85% of patients.; Orphanet notes that MRI abnormalities occur in 91% of patients but presents findings disjunctively ('corpus callosum abnormalities and/or ventriculomegaly'), indicating that the full triad is not present in every abnormal scan.","Resolve the conflict over: The structural brain MRI triad of corpus callosum hypoplasia/thinning, ventriculomegaly, and reduced white matter volume is present in over 85% of ReNU syndrome patients past infancy, providing an objective neuroradiological separator from Rett syndrome and Angelman syndrome, which typically show normal neuroimaging or non-specific cerebral volume reduction."
h13,d48,conflicted,0.72,6,2,14,0.85,False,"Hypoplasia or thinning of the corpus callosum (HP:0002079) is the predominant specific structural neuroimaging biomarker in ReNU syndrome, manifesting in approximately 85% of individuals undergoing brain MRI assessment.","contested (6 for / 2 against): Corpus callosum thinning/hypoplasia is non-specific: it is observed widely across secondary spliceosomal neurodevelopmental disorders (such as RNU2-2 and RNU5B-1) as well as non-spliceosomal microcephalic syndromic disorders.; Neuroimaging reports describe callosal hypoplasia alongside concurrent generalized cerebral volume loss, polymicrogyria, and ventriculomegaly, suggesting that callosal thinning frequently represents secondary white matter reduction rather than an isolated commissural biomarker.; Multiple diagnostic phenotyping studies identify the distinct facial gestalt and digital anomalies (e.g. hypoplastic distal phalanges) as more specific discriminative features than structural brain MRI anomalies.","Resolve the conflict over: Hypoplasia or thinning of the corpus callosum (HP:0002079) is the predominant specific structural neuroimaging biomarker in ReNU syndrome, manifesting in approximately 85% of individuals undergoing brain MRI assessment."
h7,

**Todo 4.1:** Select at least 2 refuted or conflicted hypothesis and explain why did the LLM refute/conflict the hypothesis: (1) Was the evidence truly against the hypothesis or (2) It was there missing information.

## 4.6 Report Generation

We have seen how Grill generates directions, then funds some of the promising directions and generates hyopthesis.
For each hypothesis, we try to actively look for supporting and refuting evidence. 

**Looking for refuting evidence is very important because by default LLMs actively try to look for support!**

In [18]:
from IPython.display import Markdown, display

report = (RUN_DIR / "answer.md").read_text()
print(f"{len(report):,} characters. Read it as a clinician would: what would you use to pick the patient?\n")
display(Markdown(report))

25,333 characters. Read it as a clinician would: what would you use to pick the patient?



# Clinical Identification of ReNU Syndrome (RNU4-2; OMIM #620851) from Phenotype Alone

## Executive Summary & Clinical Recognition Framework

ReNU syndrome (OMIM #620851) is an autosomal dominant neurodevelopmental disorder caused by monoallelic, predominantly de novo pathogenic variants in *RNU4-2*, which encodes the small nuclear RNA (snRNA) U4 of the major spliceosome (F1, F5, F14). Due to its high prevalence among previously unsolved neurodevelopmental delay cohorts—accounting for 0.4% to 2.9% of unexplained neurodevelopmental disorder cases and upwards of 5.3% of exome-negative cohorts (F4, F11, F14)—clinicians face the practical challenge of picking affected individuals out of patient groups presenting with non-specific severe neurodevelopmental delay, microcephaly, hypotonia, and stereotypic behaviors (F1, F4, F10).

Clinical identification rests upon a defined multi-feature constellation:
1. **A distinctive craniofacial gestalt** featuring hooded upper eyelids, full cheeks, a tented philtrum, wide mouth with downturned corners, and prominent or cupped ears (F4, F6).
2. **Congenital skeletal and orthopedic anomalies** (e.g., talipes equinovarus/clubfoot, congenital hip dysplasia) present at or near birth (F7).
3. **Early neurodevelopmental arrest with an absolute expressive speech ceiling** (absence of fluent expressive phrase speech by age 5) presenting as an early-onset static encephalopathy rather than classic Rett-like post-infancy regression (F1, F2).
4. **Selective neuroimaging patterns** demonstrating callosal thinning, delayed myelination, and ventriculomegaly in the absence of primary neuronal migration arrest (such as classical lissencephaly or subcortical band heterotopia) (F3, F8, F9).

---

## Reported Clinical Phenotypes and Cohort Frequencies

Phenotypic characterization from systematically phenotyped cohorts demonstrates high penetrance across neurobehavioral, craniofacial, growth, and musculoskeletal domains. The table below compiles frequencies reported as exact numerators and denominators with their corresponding Human Phenotype Ontology (HPO) terms.

| Phenotypic Finding (Paper Description) | Standard HPO Term | Cohort Frequency (n/N) | Percentage (%) | Resolvable Source (PMID / DOI / Citation) | Grounded Finding |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Global developmental delay / Intellectual disability** | Global developmental delay (HP:0001263) / Intellectual disability (HP:0010864) | 133 / 134<br>15 / 16 | 99.3%<br>93.8% | I. Valenzuela et al. (2024)<br>Y. Hayashi et al. (2025; DOI: [10.1038/s10038-025-01442-x](https://doi.org/10.1038/s10038-025-01442-x)) | F10 |
| **Marked expressive language deficit / speech delay** | Absent speech (HP:0000750) / Impaired expressive language (HP:0002474) | 4 / 5 (no expressive language)<br>1 / 5 (only a few words)<br>0 / 6 (words past age 5)<br>2 / 2 (evaluated deficits) | 80% (absent)<br>20% (few words)<br>0% (past age 5)<br>100% | I. Valenzuela et al. (2024)<br>N. Okamoto et al. (2025; DOI: [10.1002/ajmg.a.64151](https://doi.org/10.1002/ajmg.a.64151))<br>PF. Ajmone et al. (2026; DOI: [10.1002/ajmgb.70004](https://doi.org/10.1002/ajmgb.70004)) | F2 |
| **Microcephaly (predominantly postnatal / secondary)** | Microcephaly (HP:0000252) | 2 / 10 (congenital)<br>4 / 5 (assessed cohort)<br>2 / 2 (prenatal CNS fetuses) | ~70–85% overall<br>20% (congenital)<br>80%<br>100% (fetuses) | I. Valenzuela et al. (2024)<br>M. Barbosa et al. (2026; [PMID: 42190036](https://pubmed.ncbi.nlm.nih.gov/42190036))<br>Y. Chen et al. (2024; DOI: [10.1038/s41586-024-07773-7](https://doi.org/10.1038/s41586-024-07773-7))<br>Y. Chen et al. (2025; DOI: [10.1002/ajmgc.70002](https://doi.org/10.1002/ajmgc.70002)) | F5, F13 |
| **Characteristic facial gestalt (hooded eyelids, tented philtrum, everted lower lip)** | Distinctive facial gestalt / Abnormal facial shape (HP:0001999) | 9 / 9 | 100% | N. Okamoto et al. (2025; DOI: [10.1002/ajmg.a.64151](https://doi.org/10.1002/ajmg.a.64151)) | F4 |
| **Facial dysmorphisms: Bulbous nasal tip, cupped/prominent ears, wide mouth** | Prominent/cupped ear (HP:0000377), Bulbous nasal tip (HP:0000414), Wide mouth (HP:0000154), Downturned corners of mouth (HP:0002714) | Reported in 60–80% of assessed patients across cohorts | 60–80% | A. Bruselles et al. (2025; DOI: [10.1038/s41431-025-01820-1](https://doi.org/10.1038/s41431-025-01820-1))<br>K. Barbour et al. (2024)<br>P. Di Letto et al. (2025; DOI: [10.1212/NXG.0000000000200312](https://doi.org/10.1212/NXG.0000000000200312)) | F6 |
| **Hypotonia (generalized, axial)** | Generalized hypotonia (HP:0001252) | Major core finding across series | High penetrance | I. Valenzuela et al. (2024)<br>JM. Hussain et al. (2026; DOI: [10.34763/jmotherandchild.20263001.d-26-00014](https://doi.org/10.34763/jmotherandchild.20263001.d-26-00014)) | F1 |
| **Stereotypic hand movements** | Stereotypic hand movements (HP:0000733) | Frequent clinical presentation | Variable/Frequent | I. Valenzuela et al. (2024)<br>M. Barbosa et al. (2026) | F1 |
| **Congenital hip dysplasia / orthopedic structural anomalies** | Congenital hip dysplasia (HP:0001385) / Talipes equinovarus (HP:0001762) | 10 patients (hip dysplasia in series) | Substantial minority | M. Barbosa et al. (2026)<br>C. Nava et al. (2025; DOI: [10.1038/s41588-025-02184-4](https://doi.org/10.1038/s41588-025-02184-4)) | F7 |
| **Epilepsy / Seizures** | Seizures (HP:0001250) | 60% in Okamoto et al.<br>50% in Greene et al.<br>80% in Chen et al.<br>Overall range: 61.3–64.4% | 50–80% (age-dependent) | N. Okamoto et al. (2025; DOI: [10.1002/ajmg.a.64151](https://doi.org/10.1002/ajmg.a.64151))<br>D. Greene et al. (2024)<br>Y. Chen et al. (2024; DOI: [10.1038/s41586-024-07773-7](https://doi.org/10.1038/s41586-024-07773-7)) | F2, F12 |
| **Thinning / hypoplasia of the corpus callosum** | Hypoplasia of the corpus callosum (HP:0002079) | 10 / 11<br>4 / 14 | 90.9%<br>28.6% | I. Valenzuela et al. (2024)<br>P. Di Letto et al. (2025; DOI: [10.1212/NXG.0000000000200312](https://doi.org/10.1212/NXG.0000000000200312)) | F3, F8 |
| **Reduced white matter volume / Delayed myelination** | Decreased white matter volume (HP:0002141) / Delayed CNS myelination (HP:0002188) | 4 / 14 | 28.6% | P. Di Letto et al. (2025; DOI: [10.1212/NXG.0000000000200312](https://doi.org/10.1212/NXG.0000000000200312)) | F3, F9 |
| **Periventricular nodular heterotopia** | Periventricular nodular heterotopia (HP:0000762) | 7 / 11 (in specific subset) | <5–10% across broad cohorts | C. Nava et al. (2025)<br>I. Valenzuela et al. (2024) | F3 |

---

## Discriminating Findings vs Overlapping Phenocopies

ReNU syndrome is most commonly confused with **Rett syndrome** (*MECP2*), **Angelman syndrome** (*UBE3A* / 15q11-q13 maternal deletion), and other primary spliceosomal or microcephalic neurodevelopmental disorders (such as *RNU2-2* or *RNU5B-1* related syndromes) (F1, F4, F7, F8). 

### Comparison with Rett and Angelman Syndromes

```
+------------------------------------------------------------------------------------------------------+
|                              PHENOTYPIC DIFFERENTIATION PROFILE                                       |
+------------------------------------+----------------------------------+------------------------------+
| Phenotypic Marker                  | ReNU Syndrome (RNU4-2)           | Rett / Angelman Syndromes    |
+------------------------------------+----------------------------------+------------------------------+
| Distinctive Craniofacial Gestalt   | Hooded eyelids, full cheeks,     | Non-specific or microcephalic|
|                                    | tented philtrum, everted lip     | face (Rett); wide smiling    |
|                                    | vermilion (9/9 in Okamoto et al.)| mouth/prognathia (Angelman)  |
+------------------------------------+----------------------------------+------------------------------+
| Congenital Skeletal / Orthopedic   | Congenital hip dysplasia,        | Typically absent at birth;   |
| Malformations                      | talipes equinovarus present      | neurogenic scoliosis develops|
|                                    | near birth (substantial minority)| postnatally in mid-childhood |
+------------------------------------+----------------------------------+------------------------------+
| Developmental Trajectory           | Static early-onset arrest;       | Classic regression of acquired|
|                                    | baseline motor/social delays     | purposeful hand use/speech   |
|                                    | present from infancy             | between 6–18 mo (Rett)       |
+------------------------------------+----------------------------------+------------------------------+
| Brain MRI Pathology                | Callosal thinning, delayed       | Largely normal or non-       |
|                                    | myelination, ventriculomegaly;   | specific diffuse volume      |
|                                    | true heterotopias rare (<5-10%)  | loss without major dysgenesis|
+------------------------------------+----------------------------------+------------------------------+
```

### Confidence Assessment of Differentiation
- **High Confidence:** Presence of the specific perioral and periorbital craniofacial gestalt (hooded upper eyelids, tented philtrum, everted lower lip vermilion) reliably separates ReNU syndrome from classic *MECP2*-Rett and *UBE3A*-Angelman cohorts (F4). Furthermore, early-onset structural orthopedic signs (congenital hip dysplasia, talipes equinovarus) argue strongly against typical Rett and Angelman syndromes (F7).
- **High Confidence:** A static neurodevelopmental trajectory without classic post-infancy loss of purposeful hand skills or acquired babble separates canonical ReNU syndrome from classic Rett syndrome (F1, F15).
- **Moderate-to-Low Confidence (Spliceosomal Cohorts):** Separating ReNU syndrome from other snRNA encephalopathies (*RNU2-2*, *RNU5B-1*) cannot be accomplished by structural brain MRI alone, as thinning of the corpus callosum, ventriculomegaly, and myelination delays are shared secondary neuroimaging features across spliceosomopathies (F8, F9).

---

## Pertinent Negatives and Findings Arguing Against ReNU Syndrome

Clinical identification relies equally on evaluating pertinent negatives—findings that, if present, argue strongly against an isolated pathogenic heterozygous variant in *RNU4-2*:

1. **Fluent Expressive Language or Phrase Speech Acquisition:** Across dominant cohorts, affected children exhibit profound speech impairment; 0/6 patients acquired meaningful words past age 5 in Okamoto et al., and expressive language is universally restricted to a few single words or absent vocalizations (0% fluent sentences) (F2). The presence of fluent phrase or sentence speech effectively excludes canonical ReNU syndrome (F2).
2. **Extensive Neuronal Migration and Gyral Patterning Arrest:** Classical lissencephaly (agyria/pachygyria) or extensive subcortical band heterotopia argues against ReNU syndrome (F3). Periventricular nodular heterotopia occurs in only a small minority (<5–10% across large series) (F3), and cortical architecture in ReNU syndrome is characterized by generalized cerebral volume loss, delayed myelination, and ventriculomegaly rather than primary radial migration defects (F3, F8, F9).
3. **Post-infancy Loss of Motor Milestones (Classic Rett-like Regression):** The typical clinical course is a severe, non-progressive static encephalopathy present from the first months of life (F1, F15). A documented history of typical early development followed by classic Rett-like multi-domain loss of purposeful motor and social milestones strongly points to *MECP2* or other regressive encephalopathies rather than canonical *RNU4-2* (F1, F15).
4. **Isolated CNS Abnormalities Without Facial or Somatic Involvements:** Individuals presenting with isolated microcephaly or seizures without the characteristic dysmorphic facies, hypotonia, or developmental arrest are unlikely to harbor *RNU4-2* variants (F4, F6).

---

## Genotype-Phenotype, Positional, Ontogenetic, and Demographic Variations

### Variant Type and Position
- Pathogenic variants in *RNU4-2* are concentrated in an 18-base-pair conserved region essential for spliceosome activation, located in the 5' stem-loop and U4/U6 duplex region (F6, F11, F14).
- The recurrent single-base insertion **n.64_65insT** represents the vast majority of dominant pathogenic variants, accounting for 71.4% (20/28 in Bertoli et al., 2025) to 77.4% (89/115 in Chen et al., 2024) of alleles (F11, F14).
- Non-canonical single-nucleotide substitutions (e.g., n.66A>G, 5' stem-loop changes) have been documented to display phenotypic expansion, including atypical clinical courses with acute regression triggered by intercurrent illness or refractory epilepsy (F11, F15).
- Saturation genome editing and structural modeling indicate that variants in the T-loop produce a severe, uniform clinical presentation, whereas certain stem III variants or biallelic variants can yield differing severities (F10, F11, F12).

### Ontogenetic / Age Trajectories
- **Microcephaly:** Predominantly postnatal/secondary deceleration in cranial circumference occurs during early infancy, dropping head circumference percentiles by age 2 (F5, F13). While congenital microcephaly is noted in only 2/10 cases in Valenzuela et al., prenatal sonographic detection has been reported (2/2 fetuses in Chen et al., 2025) (F5, F13).
- **Epileptogenesis:** Seizure onset displays an age-dependent latency. In contrast to microcephaly and hypotonia (evident in infancy), epilepsy onset has a median age of approximately 3 years and can emerge into late childhood (up to 10–13 years) (F12). Consequently, younger cohorts report lower seizure frequencies (e.g., 50% in Greene et al., 2024 vs 80% in Chen et al., 2024) (F12).

### Sex
- ReNU syndrome affects males and females equally, reflecting its autosomal dominant transmission and de novo origin on non-sex chromosomes, with no reported sex-skewing in penetrance or phenotypic severity (F1, F4, F14).

---

## Contested Findings and Direct Literature Contradictions

Several hypotheses and clinical descriptions across the emerging literature exhibit direct contradictions or differing interpretations:

### 1. Neuroimaging Abnormalities: Absolute Requirement vs Non-Specific Marker
- *Contradiction:* Initial studies and smaller series reported neuroimaging abnormalities in 91% (41/45, Chen et al., 2024) to 100% (9/9, Okamoto et al., 2025; 5/5, Bruselles et al., 2025) of patients, leading some authors to consider corpus callosum thinning or abnormal MRI an obligate biomarker (F8, F9, F18).
- *Counter-evidence:* GeneReviews (Barbosa et al., 2026) reports an overall abnormal brain MRI penetrance of ~85%, and multiple cohorts demonstrate that patients with typical dysmorphic facies, microcephaly, and developmental arrest can have normal gross neuroimaging or non-specific volume loss (F8, F18). Complete absence of MRI anomalies does not exclude ReNU syndrome (F18).

### 2. Developmental Regression: Static Encephalopathy vs Intermittent Regression
- *Contradiction:* Multiple primary studies emphasize that ReNU syndrome is an early static encephalopathy lacking developmental regression, which reliably differentiates it from Rett syndrome (F1, F15).
- *Counter-evidence:* Kuroda et al. (2025) documented neurological regression in a patient carrying the non-canonical variant n.66A>G following status epilepticus or febrile illness (F15). While canonical n.64_65insT carriers generally do not display classic Rett-like regression, milestone loss can occur in the setting of severe epileptic encephalopathy or non-canonical mutations (F15).

### 3. Microcephaly: Exclusively Acquired Postnatally vs Detectable Prenatally
- *Contradiction:* Valenzuela et al. (2024) observed that microcephaly is overwhelmingly postnatal (congenital microcephaly present in only 2 of 10 assessed individuals), leading to claims that prenatal microcephaly is rare (F13).
- *Counter-evidence:* Chen et al. (2025; DOI: [10.1002/ajmgc.70002](https://doi.org/10.1002/ajmgc.70002)) screened fetuses with central nervous system anomalies and identified *RNU4-2* variants in 2 fetuses (1.35% yield), both of whom manifested microcephaly on prenatal sonography, designating microcephaly as a detectable prenatal feature in a subset of cases (F5, F13).

### 4. Recurrent n.64_65insT Allele Proportions
- *Contradiction:* Earlier estimates posited that the recurrent n.64_65insT variant accounted strictly for 70–75% of pathogenic alleles (F14).
- *Counter-evidence:* Cohort analyses show broader fluctuations, ranging from 71.4% (20/28 in Bertoli et al., 2025) up to 77.4% (89/115 in Chen et al., 2024), demonstrating that non-canonical single-base substitutions make up 22% to 29% of cases (F14).

---

## Cross-Cutting Diagnostic Synthesis and Clinical Decision Checklist

When evaluating an undiagnosed child with profound global neurodevelopmental delay, microcephaly, hypotonia, and stereotypic hand movements, clinicians should apply the following sequential bedside criteria to identify ReNU syndrome:

1. **Check Craniofacial Gestalt:** Are hooded upper eyelids, full cheeks, a tented philtrum, a wide mouth with downturned corners, and prominent or dysplastic ears present? *(Strong discriminator favoring ReNU syndrome; present in 9/9 in Okamoto et al.)* (F4, F6).
2. **Examine Skeletal and Orthopedic History:** Was there congenital hip dysplasia or talipes equinovarus at or shortly after birth? *(Discriminator against classic MECP2-Rett and UBE3A-Angelman syndromes)* (F7).
3. **Assess Expressive Speech Milestone Ceiling:** Has the patient failed to acquire phrase speech or fluent sentence production by age 5? *(Absence of phrase speech is obligatory; 0/6 acquired words past age 5)* (F2).
4. **Review Developmental Trajectory:** Did hypotonia and developmental delay begin in early infancy without a multi-month period of normal milestone acquisition followed by Rett-like developmental regression? *(Supports static ReNU trajectory; lack of regression differentiates from typical Rett)* (F1, F15).
5. **Evaluate Neuroimaging Architecture:** Does brain MRI show corpus callosum thinning, delayed myelination, or mild ventriculomegaly rather than extensive lissencephaly or subcortical band heterotopia? *(Subcortical band heterotopia or lissencephaly argues against RNU4-2)* (F3, F8, F9).

If the patient meets these criteria, targeted non-coding sequencing or reanalysis of whole-genome/extended-exome sequencing data for the *RNU4-2* 18-bp hotspot region (particularly the recurrent n.64_65insT allele) should be performed to secure the definitive molecular diagnosis (F4, F11, F14).

## Sources
1. simonssearchlight.org. RNU4-2-Related Syndrome. https://www.simonssearchlight.org/gene-guide/rnu4-2/
2. Orphanet. RNU4-2-related autosomal dominant neurodevelopmental .... Orphanet https://www.orpha.net/en/disease/detail/686488
3. epilepsygenetics.blog (2025). The Spliceosome Connection – RNU4-2 in .... https://epilepsygenetics.blog/2025/05/20/the-spliceosome-connection-rnu4-2-in-neurodevelopmental-disorders/
4. onlinelibrary.wiley.com. Deep Phenotyping in ReNU Syndrome Identifies a .... 10.1111/cge.70252
5. A Bruselles et al. (2025). Expanding the mutational spectrum of ReNU syndrome: insights into 5'Stem-loop variants. European Journal of Human Genetics 10.1038/s41431-025-01820-1
6. A Jackson et al. (2025). Analysis of R-loop forming regions identifies RNU2-2 and RNU5B-1 as neurodevelopmental disorder genes. Nature Genetics https://www.nature.com/articles/s41588-025-02209-y
7. AM Bertoli et al. (2025). RNU4-2 monoallelic variants as a leading cause of syndromic neurodevelopmental disorder, including in patients with parental consanguinity. https://jmg.bmj.com/content/62/8/536.abstract
8. B Dernek et al. (n.d.). Targeted Analysis of Spliceosomal snRNA Genes in Unexplained Neurodevelopmental Disorders Identifies ReNU Syndrome and RNU4atac-opathy. https://papers.ssrn.com/sol3/papers.cfm?abstract_id=7478549
9. C Nava et al. (2025). Dominant variants in major spliceosome U4 and U5 small nuclear RNA genes cause neurodevelopmental disorders through splicing disruption. Nature Genetics 10.1038/s41588-025-02184-4
10. D Greene et al. (2024). Mutations in the U4 snRNA gene RNU4-2 cause one of .... https://www.nature.com/articles/s41591-024-03085-5
11. I Valenzuela et al. (2024). Deep phenotyping of 11 individuals with pathogenic variants in RNU4-2 reveals a clinically recognizable syndrome. https://www.sciencedirect.com/science/article/pii/S1098360024002223
12. J De Jonghe et al. (2026). Saturation editing of RNU4-2 reveals distinct dominant and recessive disorders. 10.1038/s41586-026-10334-9
13. J Rosenblum et al. (2025). RNU4‐2‐related neurodevelopmental disorder is associated with a recognisable facial gestalt. 10.1111/cge.14628
14. JM Hussain et al. (2026). ReNU Syndrome and the RNU4-2 Mutation: A Missed Cause of Childhood Neurodevelopmental Delay. 10.34763/jmotherandchild.20263001.d-26-00014
15. K Barbour et al. (2024). The face and features of RNU4-2: a new, common, recognizable, yet hidden neurodevelopmental disorder. Pediatr Neurol 10.1016/j.pediatrneurol.2024.09.015
16. K Barbour et al. (2024). The Face and Features of RNU4-2 - Pediatric Neurology. Pediatric Neurology https://www.pedneur.com/article/S0887-8994(24)00340-0/fulltext
17. M Barbosa et al. (2026). RNU4-2–Related Autosomal Dominant Neurodevelopmental Disorder. GeneReviews https://www.ncbi.nlm.nih.gov/books/NBK622730/
18. M Mastrangelo et al. (2026). Epilepsy phenotypes of Renu syndrome: Novel insights from a European multicentre retrospective cohort study. Seizure 10.1016/j.seizure.2026.01.010
19. M Sevilla et al. (2025). Characterization of snRNA-related neurodevelopmental disorders through the Spanish Undiagnosed Rare Disease Programs. medRxiv 10.1101/2025.09.16.25335449.abstract
20. N Löker et al. (n.d.). 10th Erciyes Medical Genetics Congress with International Participation. https://www.researchgate.net/profile/Behich-Koyutourk/publication/394443709_A_de_novo_pathogenic_FOXG1_frameshift_mutation_associated_with_autosomal_dominant_congenital_Rett_Syndrome/links/689af78ba49b125ba30cb586/A-de-novo-pathogenic-FOXG1-frameshift-mutation-associated-with-autosomal-dominant-congenital-Rett-Syndrome.pdf
21. N Okamoto et al. (2025). A clinical study of nine patients with ReNU syndrome. Am J Med Genet A 10.1002/ajmg.a.64151
22. NCBI ClinVar et al. (n.d.). NR_003137.3(RNU4-2):n.64_65insT - NCBI. ClinVar https://www.ncbi.nlm.nih.gov/clinvar/variation/3068742/
23. P Di Letto et al. (2025). Reanalysis of Undiagnosed Neurodevelopmental Disorder Cases: From RNU4-2 Variants to Clinical Phenotypes. Neurology: Genetics 10.1212/NXG.0000000000200312
24. PF Ajmone et al. (2026). Longitudinal Behavior Phenotype Hallmarks in RNU4‐2 Syndrome: Implications for Clinical Management. 10.1002/ajmgb.70004
25. R Rius et al. (2026). Biallelic variants in the noncoding RNA gene RNU4-2 cause a recessive neurodevelopmental syndrome with distinct white matter changes. Nature Genetics 10.1038/s41588-026-02554-6
26. R Rius et al. (2025). Biallelic variants in the non-coding RNA gene RNU4-2 cause a novel recessive neurodevelopmental syndrome. medRxiv https://www.medrxiv.org/content/medrxiv/early/2025/08/16/2025.08.13.25333306.full.pdf
27. SE Antonarakis et al. (2026). Small nuclear RNA genes in Mendelian disorders. Nature Genetics 10.1038/s41588-025-02440-7
28. T Hiraide et al. (2026). Recurrent RNU4-2 n. 64_65insT variant in ReNU syndrome identified in exome-negative cases. 10.1016/j.braindev.2026.104570
29. Y Chen et al. (2024). De novo variants in the RNU4-2 snRNA cause a frequent .... Nature 10.1038/s41586-024-07773-7
30. Y Chen et al. (2025). Prenatal Evaluation of RNU4‐2 Variants in Fetuses With Central Nervous System Anomalies. American Journal of Medical Genetics Part C 10.1002/ajmgc.70002
31. Y Chen et al. (2026). Recurrent RNU4-2 n.64_65insT variant in ReNU syndrome .... Brain and Development https://www.sciencedirect.com/science/article/abs/pii/S0387760426000719
32. Y Chen et al. (2024). De novo variants in the RNU4-2 snRNA cause a frequent neurodevelopmental syndrome. Nature 10.1038/s41586-024-07773-7
33. Y Chen et al. (2024). De novo variants in the non-coding spliceosomal snRNA gene RNU4-2 are a frequent cause of syndromic neurodevelopmental disorders. PMC11030480
34. Y Hayashi et al. (2025). Monoallelic and biallelic RNU4-2 variants in neurodevelopmental disorders. Journal of Human Genetics 10.1038/s10038-025-01442-x
35. Y Kuroda et al. (2025). Genotype-phenotype correlations and phenotypic expansion in a case series of ReNU syndrome associated with RNU4-2 variants. 10.1136/jmg-2024-110604
36. Y Nakano et al. (2026). Whole-genome discovery of pathogenic snRNA variants and efficient extended-exome screening. 10.1016/j.isci.2026.116814
37. Z Saleem et al. (2026). Correspondence on “Deep phenotyping of 11 individuals with pathogenic variants in RNU4-2 reveals a clinically recognizable syndrome” by Valenzuela et al. Genetics in Medicine https://www.gimjournal.org/article/S1098-3600(26)00004-3/abstract


**TODO 4.2:** Even thought the GRILL report is big, is there  any thing that it misses compared to the report generated by the RAG system or direct LLM call in sec 1.3 and 1.4?

**TODO 4.3:**: Given this report, and the patient cohort data can you guess which patient has this disease?

# 5 Ranking the patients

We will now rank the patient according to the generated grill report.

In [22]:
report_run = json.loads((ROOT / "assignments" / DISEASE.replace(":", "_") / "report_match.json").read_text())
report_messages = report_run["messages"]

print(f"Same task as before, same ten patients, same output shape. Only the evidence changes:")
print(f"the six snippets are replaced by GRILL's report, {report_run['report_chars']:,} characters.\n")
for m in report_messages:
    print(f"--- {m['role']} ({len(m['content']):,} chars) ---")
    body = m["content"]
    if m["role"] == "user":
        head, _, tail = body.partition("COHORT (")
        body = head[:1200] + "\n  [... the rest of the report ...]\n\nCOHORT (" + tail[:900] + "\n  [... remaining patients ...]"
    print(textwrap.indent(body, "  "))
    print()

Same task as before, same ten patients, same output shape. Only the evidence changes:
the six snippets are replaced by GRILL's report, 25,333 characters.

--- system (335 chars) ---
  You are a clinical geneticist. You are given a literature report on one rare disease and a cohort of patients described by HPO terms. Exactly one patient has the disease. Use only the report as your knowledge of the disease. Be explicit about which findings in the report drove each placement, and about findings that argue against it.

--- user (32,091 chars) ---
  REPORT ON ReNU syndrome (RNU4-2, OMIM:620851):

  # Clinical Identification of ReNU Syndrome (RNU4-2; OMIM #620851) from Phenotype Alone

  ## Executive Summary & Clinical Recognition Framework

  ReNU syndrome (OMIM #620851) is an autosomal dominant neurodevelopmental disorder caused by monoallelic, predominantly de novo pathogenic variants in *RNU4-2*, which encodes the small nuclear RNA (snRNA) U4 of the major spliceosome (F1, F5, F14). Due t

## 5.1 Running with LLM

We again send the same report to the LLM three times, and we might again get different ranking each time.

In [24]:
rows = []
for r in report_run["rounds"]:
    for pos, c in enumerate(r["ranking"][:5], 1):
        rows.append({"round": r["round"], "position": pos, "patient": c["patient_id"],
                     "probability": c["probability"], "for": c["for"], "against": c["against"]})
report_ranking = pd.DataFrame(rows).set_index(["round", "position"])

display(report_ranking.style
          .set_properties(subset=["for", "against"],
                          **{"white-space": "normal", "text-align": "left", "max-width": "420px"})
          .format({"probability": "{:.2f}"})
          .set_caption(f"three rounds with the GRILL report: the model's top five for {brief.label}"))

**TODO 5.1:** Did repeated runs agree? They if do not disagree, do their reasoning match, please describe. If they do not match, how does their reasoning differ?

**TODO 5.2:** What is the differentiating factor in each round between the first and the second ranks?

**TODO 5.3:** How does the ranking differ from RAG generated ranking in section 3?

**TODO 5.4:** Based on the information gather, which patient is likely to have the disease (take your best guess!)

# 6 Evaluation

Now you have seen how simple RAG works, how with a more extensive method of searching we can enrich the context. But how do we tell which one is correct?

Fortunately, for this task we have the ground truth. After running the cell below, we will learn the answer and the ranking from RAG and Grill report.

In [26]:
import csv

key_path = ROOT / "data" / "assignment_targets.csv"
if not key_path.exists():
    print("No answer key in this copy. Submit your ranking; it will be scored for you.")
else:
    my_dir = DISEASE.replace(":", "_")
    row = next(r for r in csv.DictReader(key_path.open()) if r["directory"] == my_dir)
    TARGET_PATIENT = row["target_patient_id"]

    print(f"{row['disease']} ({row['gene']}): the patient with the disease is {TARGET_PATIENT}")
    print(f"source case report: PMID:{row['source_pmid']}   phenopacket: {row['phenopacket']}\n")

    def rank_of(run):
        out = []
        for r in run["rounds"]:
            ids = [c["patient_id"] for c in r["ranking"][:5]]
            out.append(ids.index(TARGET_PATIENT) + 1 if TARGET_PATIENT in ids else None)
        return out

    scored = pd.DataFrame({
        "RAG":  rank_of(rank_run),
        "GRILL report":  rank_of(report_run),
    }, index=[f"round {i}" for i in (1, 2, 3)])
    display(scored.style.format(lambda v: "not in top 5" if v is None or pd.isna(v) else f"rank {int(v)}")
                        .set_caption(f"where {TARGET_PATIENT} landed in each ranking"))

    # The true patient, as you saw them.
    p = cohort.get(TARGET_PATIENT)
    print(f"\n{TARGET_PATIENT}: age {p.age_years}, {p.sex}, {len(p.terms)} findings, {len(p.excluded)} recorded absent")
    for t in p.terms:
        print(f"    {t}  {hpo.label(t)}")

ReNU syndrome (RNU4-2): the patient with the disease is P06
source case report: PMID:39369315   phenopacket: PMID_39369315_Participant_9



,RAG,GRILL report
round 1,rank 1,rank 1
round 2,rank 1,rank 1
round 3,rank 1,rank 1



P06: age 1.7, male, 31 findings, 40 recorded absent
    HP:0033454  Tube feeding
    HP:0001263  Global developmental delay
    HP:0000639  Nystagmus
    HP:0001508  Failure to thrive
    HP:0002019  Constipation
    HP:0000253  Progressive microcephaly
    HP:0001344  Absent speech
    HP:0001270  Motor delay
    HP:0000873  Diabetes insipidus
    HP:0000729  Autistic behavior
    HP:0000733  Motor stereotypy
    HP:0002079  Hypoplasia of the corpus callosum
    HP:0002188  Delayed CNS myelination
    HP:0002307  Drooling
    HP:0000737  Irritability
    HP:0001288  Gait disturbance
    HP:0000194  Open mouth
    HP:0010804  Tented upper lip vermilion
    HP:0011825  Tented philtrum
    HP:0000286  Epicanthus
    HP:0000297  Facial hypotonia
    HP:0000581  Blepharophimosis
    HP:0011228  Horizontal eyebrow
    HP:0000348  High forehead
    HP:0000385  Small earlobe
    HP:0000341  Narrow forehead
    HP:0006532  Recurrent pneumonia
    HP:0031061  Impaired toileting ability
    HP:

## 6.1 Analysing Failures

Write your code for evaluating why did RAG or GRILL get the ranking wrong.

For grill:
- Assignment (Root)
  - `grill_run/`
    - `tasks/`  # contains step wise output information and associated metadata.
    - `ledger.json`  # detailed information on what the agent was trying to do
    -  ...

We recommend going through `tasks` to understand what the agent was getting wrong.

We also recommend looking at the `ledger.json`, so that you understand the reasoning, direction, hypothesis, budget etc that the agent was trying to do.

**TODO 6.1: Write your code below for analysis and then write your findings on why did any system fail.**